In [ ]:
NOTEBOOK_SOURCE_SHA256 = '9a73b74d6bc97f2f0ebf27b56807e70f6035805a153d25dfd2097624be4e45e0'
import os, sys, json, time, threading, subprocess, hashlib
from pathlib import Path
from datetime import datetime, timezone
from contextlib import contextmanager
def event(stage, status):
    print(datetime.now(timezone.utc).isoformat(), stage, status, flush=True)
@contextmanager
def stage(name):
    event(name, 'start')
    stop = threading.Event()
    def pulse():
        while not stop.wait(5): event(name, 'heartbeat')
    thread = threading.Thread(target=pulse, daemon=True)
    thread.start()
    try:
        yield
    except BaseException:
        event(name, 'failed')
        raise
    else:
        event(name, 'done')
    finally:
        stop.set()
        thread.join()
event('startup/config', 'start')
os.environ.update(CUDA_VISIBLE_DEVICES='0', PYTHONUNBUFFERED='1', HF_HUB_DOWNLOAD_TIMEOUT='60', HF_HUB_ETAG_TIMEOUT='60')
OUT = Path('/kaggle/working/outputs/benchmarks') / ('smoke10-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
OUT.mkdir(parents=True, exist_ok=False)
MODEL = 'vinai/PhoWhisper-base'
def command(argv, log, timeout=None):
    with stage(log), (OUT / log).open('w') as stream:
        child = subprocess.Popen(argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=os.environ.copy())
        timer = threading.Timer(timeout, child.kill) if timeout else None
        if timer: timer.start()
        try:
            for line in child.stdout:
                print(line, end='', flush=True)
                stream.write(line); stream.flush()
            if child.wait(): raise RuntimeError('Command failed: ' + log)
        finally:
            if timer: timer.cancel()
            if child.poll() is None: child.kill(); child.wait()
event('startup/config', 'done')


In [ ]:
with stage('dependencies/versions'):
    import importlib, importlib.util
    if importlib.util.find_spec('jiwer') is None:
        command([sys.executable, '-u', '-m', 'pip', 'install', '--retries', '2', '--timeout', '30', 'jiwer==3.0.5'], 'pip.log', 600)
    from importlib.metadata import version
    versions = {}
    for name in ('torch', 'transformers', 'accelerate', 'huggingface_hub', 'numpy', 'scipy', 'soundfile', 'jiwer'):
        try: importlib.import_module(name)
        except ImportError as exc: raise RuntimeError('Fix image dependency: ' + name) from exc
        versions[name] = version(name)
    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError('Expected one visible CUDA GPU')
    print(versions, flush=True)
    (OUT / 'dependencies.json').write_text(json.dumps(versions, indent=2))


In [ ]:
"""Metadata-only selection; audio audit is restricted to the selected ten files."""
import csv
import random
import re
import unicodedata
from collections import defaultdict
from pathlib import Path


def metadata(root, split):
    root = Path(root).resolve()
    rows, seen = [], set()
    with (root / f'metadata_{split}.csv').open(encoding='utf-8', newline='') as stream:
        reader = csv.DictReader(stream, delimiter='|')
        if not {'audio_file', 'text', 'speaker_name'} <= set(reader.fieldnames or []):
            raise ValueError('Required columns: audio_file|text|speaker_name')
        for row in reader:
            name = row['audio_file']
            if not re.fullmatch(r'(?:wavs/)?audio\d+\.wav', name):
                raise ValueError('Unsafe metadata path')
            name = Path(name).name
            if name in seen:
                raise ValueError('Duplicate metadata ID')
            seen.add(name)
            speaker = ' '.join(unicodedata.normalize('NFC', row['speaker_name']).casefold().split())
            if not speaker or not row['text'].strip():
                raise ValueError('Empty speaker/text')
            rows.append(dict(id=name, speaker=speaker, text=row['text'],
                             audio_path=str(root / 'wavs' / name)))
    return rows


def select(rows, test_ids, seed=42):
    rng = random.Random(seed)
    buckets = defaultdict(list)
    for row in rows:
        if int(Path(row['id']).stem[5:]) > 1250 and row['id'] not in test_ids:
            buckets[row['speaker']].append(row)
    speakers = sorted(buckets)
    rng.shuffle(speakers)
    for bucket in buckets.values():
        rng.shuffle(bucket)
    reserved, count = [], 0
    for speaker in speakers:
        reserved.append(speaker)
        count += len(buckets[speaker])
        if count >= 2:
            break
    def balanced(names, size):
        result = []
        for index in range(max((len(buckets[s]) for s in names), default=0)):
            for speaker in names:
                if index < len(buckets[speaker]):
                    result.append(buckets[speaker][index])
                    if len(result) == size:
                        return result
        raise ValueError('Insufficient speaker-disjoint data')
    return balanced([s for s in speakers if s not in reserved], 8), balanced(reserved, 2)


def audit(rows, root, processor):
    import numpy as np
    import soundfile as sf
    import hashlib
    hashes = set()
    root = Path(root).resolve()
    for row in rows:
        path = Path(row['audio_path']).resolve()
        if not path.is_relative_to(root) or not path.is_file():
            raise ValueError('Missing/unsafe selected audio')
        info = sf.info(path)
        if info.frames <= 0 or info.samplerate <= 0 or not 0 < info.duration <= 30:
            raise ValueError('Invalid selected audio duration')
        audio, _ = sf.read(path, dtype='float32', always_2d=True)
        if not np.isfinite(audio).all():
            raise ValueError('Nonfinite selected audio')
        if not 0 < len(processor.tokenizer(row['text']).input_ids) <= 448:
            raise ValueError('Invalid label length')
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        row['sha256'] = digest.hexdigest()
        if row['sha256'] in hashes:
            raise ValueError('Duplicate selected audio bytes')
        hashes.add(row['sha256'])

with stage('metadata/subset/audio-validation'):
    roots = [Path('/kaggle/input/datasets/tuandaodev/viet-bud500-formatted'), Path('/kaggle/input/viet-bud500-formatted')]
    ROOT = next((p for p in roots if all((p / ('metadata_' + s + '.csv')).is_file() for s in ('train', 'test'))), None)
    if ROOT is None: raise ValueError('Expected exact dataset metadata layout')
    rows, official = metadata(ROOT, 'train'), metadata(ROOT, 'test')
    train_rows, val_rows = select(rows, {r['id'] for r in official})
    from huggingface_hub import model_info
    from transformers import WhisperProcessor
    REVISION = model_info(MODEL, timeout=60).sha
    processor = WhisperProcessor.from_pretrained(MODEL, revision=REVISION, language='vi', task='transcribe')
    audit(train_rows + val_rows, ROOT, processor)
    for name, subset in [('train', train_rows), ('validation', val_rows)]:
        (OUT / (name + '.json')).write_text(json.dumps(subset, ensure_ascii=False, indent=2))
    COUNTS = dict(train=8, validation=2, corpus_speakers=len({r['speaker'] for r in rows}),
                  train_speakers=len({r['speaker'] for r in train_rows}), validation_speakers=len({r['speaker'] for r in val_rows}), speaker_disjoint=not bool({r['speaker'] for r in train_rows} & {r['speaker'] for r in val_rows}))
    print(COUNTS, flush=True)


In [ ]:
with stage('emit-trainer'):
    TRAINER_SOURCE = '"""Standalone, lazy-audio PhoWhisper training. No inference-runtime imports."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport os\nimport random\nimport re\nimport unicodedata\nfrom pathlib import Path\nimport time\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\n\n\nclass Progress:\n    def __init__(self, path=None, current=None):\n        self.path = Path(path) if path else None\n        self.current = Path(current) if current else None\n        self.last_progress = None\n\n    def update(self, stage, status=\'running\', **values):\n        record = dict(stage=stage, status=status, time=datetime.now(timezone.utc).isoformat(), **values)\n        if \'processed\' in values:\n            self.last_progress = dict(stage=stage, time=record[\'time\'], **values)\n        record[\'last_progress\'] = self.last_progress\n        print(json.dumps(record), flush=True)\n        for path in (self.path, self.current):\n            if path:\n                path.parent.mkdir(parents=True, exist_ok=True)\n                temporary = path.with_name(path.name + f\'.{os.getpid()}.tmp\')\n                temporary.write_text(json.dumps(record, indent=2), encoding=\'utf-8\')\n                os.replace(temporary, path)\n\n    @contextmanager\n    def stage(self, name):\n        started = time.monotonic()\n        self.update(name, \'start\')\n        try:\n            yield\n        except BaseException as exc:\n            self.update(name, \'failed\', elapsed_seconds=time.monotonic() - started,\n                        error_type=type(exc).__name__)\n            raise\n        else:\n            self.update(name, \'done\', elapsed_seconds=time.monotonic() - started)\n\n\nclass StepProgress:\n    def __init__(self, progress, clock=time.monotonic):\n        self.progress, self.clock = progress, clock\n\n    def on_train_begin(self, args, state, control, **kwargs):\n        self.warm_time, self.warm_step = None, None\n        self.progress.update(\'optimizer\', processed=state.global_step, total=state.max_steps)\n\n    def on_step_end(self, args, state, control, **kwargs):\n        now = self.clock()\n        if self.warm_time is None and state.global_step >= 5:\n            self.warm_time, self.warm_step = now, state.global_step\n        eta = None\n        if self.warm_time is not None and state.global_step > self.warm_step:\n            eta = (now - self.warm_time) / (state.global_step - self.warm_step) * max(0, state.max_steps - state.global_step)\n        self.progress.update(\'optimizer\', processed=state.global_step, total=state.max_steps,\n            eta_seconds=eta, eta_scope=\'training optimizer steps only; excludes evaluation/save/reporting\')\n\n\ndef normalize(text):\n    text = unicodedata.normalize(\'NFC\', text).lower()\n    return \' \'.join(\'\'.join(\' \' if unicodedata.category(c).startswith(\'P\') else c for c in text).split())\n\n\ndef read_metadata(root, split):\n    root = Path(root).resolve()\n    rows, seen = [], set()\n    with (root / f\'metadata_{split}.csv\').open(encoding=\'utf-8\', newline=\'\') as stream:\n        reader = csv.DictReader(stream, delimiter=\'|\')\n        if not {\'audio_file\', \'text\', \'speaker_name\'} <= set(reader.fieldnames or []):\n            raise ValueError(\'Required columns: audio_file|text|speaker_name\')\n        for row in reader:\n            name = row[\'audio_file\']\n            if not re.fullmatch(r\'(?:wavs/)?audio\\d+\\.wav\', name):\n                raise ValueError(f\'Unexpected path: {name}\')\n            name = Path(name).name\n            if name in seen or not row[\'text\'].strip():\n                raise ValueError(f\'Duplicate ID or empty text: {name}\')\n            seen.add(name)\n            path = root / \'wavs\' / name\n            if not path.is_file() or not path.resolve().is_relative_to(root):\n                raise ValueError(f\'Missing/unsafe audio: {path}\')\n            rows.append(dict(id=name, audio_path=str(path), text=row[\'text\'], speaker=row[\'speaker_name\'].strip()))\n    return rows\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef prepare(args):\n    train, test = read_metadata(args.root, \'train\'), read_metadata(args.root, \'test\')\n    if {r[\'id\'] for r in train} & {r[\'id\'] for r in test}:\n        raise ValueError(\'Official train/test ID overlap; resolve provenance before training\')\n    excluded_ids, excluded_hashes = set(), set()\n    for manifest in args.exclude_manifest:\n        with open(manifest, encoding=\'utf-8\') as stream:\n            for row in csv.DictReader(stream):\n                excluded_ids.add(Path(row.get(\'audio_path\', row.get(\'id\', \'\'))).stem)\n                if row.get(\'sha256\'):\n                    excluded_hashes.add(row[\'sha256\'])\n    kept, seen_hashes, excluded = [], set(), 0\n    for row in train:\n        number = int(Path(row[\'id\']).stem[5:])\n        if number <= 1250 or Path(row[\'id\']).stem in excluded_ids:\n            excluded += 1\n            continue\n        row[\'sha256\'] = digest(row[\'audio_path\'])\n        if row[\'sha256\'] in excluded_hashes:\n            excluded += 1\n            continue\n        if row[\'sha256\'] in seen_hashes:\n            raise ValueError(f\'Duplicate audio bytes: {row["id"]}\')\n        seen_hashes.add(row[\'sha256\'])\n        kept.append(row)\n    rng = random.Random(args.seed)\n    speakers = sorted({r[\'speaker\'] for r in kept})\n    if \'\' in speakers or len(speakers) < 2:\n        raise ValueError(\'Speaker-group split requires at least two nonempty speaker IDs\')\n    rng.shuffle(speakers)\n    validation_speakers = set(speakers[:max(1, int(len(speakers) * args.validation_fraction))])\n    groups = {\'train\': [r for r in kept if r[\'speaker\'] not in validation_speakers],\n              \'validation\': [r for r in kept if r[\'speaker\'] in validation_speakers], \'test\': test}\n    for name, rows in groups.items():\n        if name != \'test\':\n            rng.shuffle(rows)\n            cap = args.train_cap if name == \'train\' else args.validation_cap\n            rows = rows[:cap] if cap else rows\n        if not rows:\n            raise ValueError(f\'Empty {name} split\')\n        (args.output / f\'{name}.json\').write_text(json.dumps(rows, ensure_ascii=False, indent=2))\n    (args.output / \'prepare.json\').write_text(json.dumps(dict(seed=args.seed, root=str(args.root),\n        excluded=excluded, train_cap=args.train_cap, validation_cap=args.validation_cap,\n        validation_speakers=sorted(validation_speakers), exclude_manifest=args.exclude_manifest), indent=2))\n\n\nclass AudioDataset:\n    def __init__(self, rows, processor):\n        self.rows, self.processor = rows, processor\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        import numpy as np\n        import soundfile as sf\n        from scipy.signal import resample_poly\n        from math import gcd\n        row = self.rows[index]\n        info = sf.info(row[\'audio_path\'])\n        if info.frames <= 0 or info.samplerate <= 0 or info.duration > 30:\n            raise ValueError(f\'Empty/invalid/>30s audio: {row["id"]}\')\n        audio, rate = sf.read(row[\'audio_path\'], dtype=\'float32\', always_2d=True)\n        audio = audio.mean(axis=1)\n        if not np.isfinite(audio).all():\n            raise ValueError(f\'Nonfinite audio: {row["id"]}\')\n        factor = gcd(rate, 16000)\n        audio = resample_poly(audio, 16000 // factor, rate // factor)\n        labels = self.processor.tokenizer(row[\'text\']).input_ids\n        if len(labels) > 448:\n            raise ValueError(f\'Transcript exceeds decoder capacity: {row["id"]}\')\n        return {\'input_features\': self.processor.feature_extractor(audio, sampling_rate=16000).input_features[0],\n                \'labels\': labels}\n\n\nclass Collator:\n    def __init__(self, processor, start):\n        self.processor, self.start = processor, start\n\n    def __call__(self, features):\n        batch = self.processor.feature_extractor.pad([{\'input_features\': f[\'input_features\']} for f in features], return_tensors=\'pt\')\n        # Strip per example, never remove a real first text token.\n        labels = []\n        for feature in features:\n            tokens = feature[\'labels\']\n            tokens = tokens[1:] if tokens and tokens[0] == self.start else tokens\n            if not tokens:\n                raise ValueError(\'Empty decoder labels after decoder-start stripping\')\n            labels.append(tokens)\n        padded = self.processor.tokenizer.pad([{\'input_ids\': x} for x in labels], return_tensors=\'pt\')\n        batch[\'labels\'] = padded.input_ids.masked_fill(padded.attention_mask.ne(1), -100)\n        return batch\n\n\ndef run(args):\n    progress = Progress(os.environ.get(\'PILOT_PROGRESS\'), os.environ.get(\'PILOT_CURRENT_PROGRESS\'))\n    progress.update(\'training_dependency_imports\', \'start\')\n    import numpy as np\n    import torch\n    import jiwer\n    from huggingface_hub import model_info\n    from transformers import (WhisperProcessor, WhisperForConditionalGeneration,\n                              Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback, set_seed)\n    set_seed(args.seed)\n    progress.update(\'training_dependency_imports\', \'done\')\n    source = args.model\n    with progress.stage(\'training_hf_metadata\'):\n        revision = args.revision if Path(source).is_dir() else model_info(source, revision=args.revision, timeout=60).sha\n    with progress.stage(\'training_processor_load\'):\n        processor = WhisperProcessor.from_pretrained(source, revision=revision, language=\'vi\', task=\'transcribe\')\n    processor.tokenizer.set_prefix_tokens(language=\'vi\', task=\'transcribe\', predict_timestamps=False)\n    with progress.stage(\'training_model_load\'):\n        model = WhisperForConditionalGeneration.from_pretrained(source, revision=revision)\n    if args.precision == \'fp32\':\n        model.float()\n    model.config.use_cache = False\n    model.config.forced_decoder_ids = None\n    model.generation_config.forced_decoder_ids = None\n    model.generation_config.language = \'vi\'\n    model.generation_config.task = \'transcribe\'\n    model.generation_config.return_timestamps = False\n    model.generation_config.num_beams = 1\n    model.generation_config.do_sample = False\n    def metrics(prediction):\n        labels = np.where(prediction.label_ids == -100, processor.tokenizer.pad_token_id, prediction.label_ids)\n        refs = [normalize(x) for x in processor.batch_decode(labels, skip_special_tokens=True)]\n        hyps = [normalize(x) for x in processor.batch_decode(prediction.predictions, skip_special_tokens=True)]\n        words = jiwer.process_words(refs, hyps)\n        return dict(wer=words.wer, cer=jiwer.cer(refs, hyps), substitutions=words.substitutions,\n                    deletions=words.deletions, insertions=words.insertions)\n    rows = json.loads(args.manifest.read_text())\n    dataset = AudioDataset(rows, processor)\n    training = args.mode in (\'train\', \'smoke\')\n    validation = AudioDataset(json.loads(args.validation.read_text()), processor) if training else dataset\n    options = Seq2SeqTrainingArguments(output_dir=str(args.output), seed=args.seed,\n        per_device_train_batch_size=args.batch, per_device_eval_batch_size=1,\n        gradient_accumulation_steps=args.accumulation, learning_rate=args.learning_rate,\n        max_steps=args.max_steps, fp16=(args.precision == \'fp16\' or\n            (args.precision == \'auto\' and torch.cuda.is_available())),\n        bf16=args.precision == \'bf16\', gradient_checkpointing=True,\n        gradient_checkpointing_kwargs={\'use_reentrant\': False},\n        eval_strategy=\'steps\' if training else \'no\', eval_steps=args.save_steps,\n        save_steps=args.save_steps, save_total_limit=2, logging_steps=1,\n        predict_with_generate=True, generation_max_length=448, generation_num_beams=1,\n        dataloader_num_workers=args.workers, report_to=[], remove_unused_columns=False,\n        ddp_find_unused_parameters=False)\n    trainer = Seq2SeqTrainer(model=model, args=options, train_dataset=dataset if training else None,\n        eval_dataset=validation, data_collator=Collator(processor, model.config.decoder_start_token_id),\n        processing_class=processor, compute_metrics=metrics)\n    class OptimizerProgress(StepProgress, TrainerCallback):\n        pass\n    trainer.add_callback(OptimizerProgress(progress))\n    if args.require_finite_training:\n        class FiniteTraining(TrainerCallback):\n            def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):\n                if any(p.grad is not None and not torch.isfinite(p.grad).all() for p in model.parameters()):\n                    raise ValueError(\'Nonfinite gradient; pilot aborted\')\n\n            def on_step_end(self, args, state, control, model=None, **kwargs):\n                if any(not torch.isfinite(p).all() for p in model.parameters()):\n                    raise ValueError(\'Nonfinite model parameter; pilot aborted\')\n        trainer.add_callback(FiniteTraining())\n    if trainer.is_world_process_zero():\n        processor.save_pretrained(args.output)\n        (args.output / \'run.json\').write_text(json.dumps(dict(arguments={k: str(v) for k, v in vars(args).items()},\n            resolved_revision=revision, world_size=int(os.environ.get(\'WORLD_SIZE\', 1)),\n            manifest_sha256=digest(args.manifest), validation_sha256=digest(args.validation) if args.validation else None,\n            torch=torch.__version__), indent=2))\n    if training:\n        with progress.stage(\'trainer_train\'):\n            trainer.train(resume_from_checkpoint=args.resume or None)\n        trainer.save_model()\n        trainer.save_state()\n    with progress.stage(\'trainer_prediction\'):\n        prediction = trainer.predict(validation)\n    if trainer.is_world_process_zero():\n        (args.output / \'metrics.json\').write_text(json.dumps(prediction.metrics, indent=2))\n        hyp = processor.batch_decode(prediction.predictions, skip_special_tokens=True)\n        with (args.output / \'errors.csv\').open(\'w\', encoding=\'utf-8\', newline=\'\') as stream:\n            writer = csv.writer(stream)\n            writer.writerow([\'id\', \'reference\', \'hypothesis\', \'substitutions\', \'deletions\', \'insertions\'])\n            for row, text in zip(validation.rows, hyp):\n                result = jiwer.process_words(normalize(row[\'text\']), normalize(text))\n                writer.writerow([row[\'id\'], row[\'text\'], text, result.substitutions, result.deletions, result.insertions])\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument(\'mode\', choices=[\'inspect\', \'prepare\', \'smoke\', \'train\', \'evaluate\'])\n    p.add_argument(\'--root\', type=Path)\n    p.add_argument(\'--output\', type=Path, required=True)\n    p.add_argument(\'--exclude-manifest\', action=\'append\', default=[])\n    p.add_argument(\'--seed\', type=int, default=42)\n    p.add_argument(\'--validation-fraction\', type=float, default=.05)\n    p.add_argument(\'--train-cap\', type=int, default=2000)\n    p.add_argument(\'--validation-cap\', type=int, default=100)\n    p.add_argument(\'--manifest\', type=Path)\n    p.add_argument(\'--validation\', type=Path)\n    p.add_argument(\'--model\', default=\'vinai/PhoWhisper-base\')\n    p.add_argument(\'--revision\', default=\'main\')\n    p.add_argument(\'--resume\')\n    p.add_argument(\'--max-steps\', type=int, default=10)\n    p.add_argument(\'--save-steps\', type=int, default=100)\n    p.add_argument(\'--batch\', type=int, default=2)\n    p.add_argument(\'--accumulation\', type=int, default=4)\n    p.add_argument(\'--workers\', type=int, default=0)\n    p.add_argument(\'--learning-rate\', type=float, default=1e-5)\n    p.add_argument(\'--precision\', choices=[\'auto\', \'fp32\', \'fp16\', \'bf16\'], default=\'auto\')\n    p.add_argument(\'--require-finite-training\', action=\'store_true\')\n    args = p.parse_args()\n    if min(args.train_cap, args.validation_cap, args.workers) < 0 or min(args.max_steps, args.save_steps, args.batch, args.accumulation) <= 0:\n        p.error(\'caps/workers must be nonnegative; steps/batch/accumulation positive\')\n    if args.mode in (\'inspect\', \'prepare\'):\n        if not args.root or not 0 < args.validation_fraction < 1:\n            p.error(\'root and validation fraction in (0,1) required\')\n        if args.mode == \'inspect\':\n            print(json.dumps({s: len(read_metadata(args.root, s)) for s in (\'train\', \'test\')}))\n            return\n    elif not args.manifest or (args.mode in (\'train\', \'smoke\') and not args.validation):\n        p.error(\'manifest required; training also needs validation\')\n    if args.mode in (\'inspect\', \'prepare\') and int(os.environ.get(\'WORLD_SIZE\', 1)) > 1:\n        p.error(\'prepare/inspect must run on one process\')\n    args.output.mkdir(parents=True, exist_ok=True)\n    prepare(args) if args.mode == \'prepare\' else run(args)\n\n\nif __name__ == \'__main__\':\n    main()\n'
    TRAINER = OUT / 'train.py'
    TRAINER.write_text(TRAINER_SOURCE)
    PROVENANCE = dict(base_git_revision='b15d7fb0141f385fd4bc6014df915e1c37212467', executed_git_revision=None,
        trainer_sha256=hashlib.sha256(TRAINER_SOURCE.encode()).hexdigest(), resolved_revision=REVISION,
        notebook_source_sha256=NOTEBOOK_SOURCE_SHA256)
    (OUT / 'provenance.json').write_text(json.dumps(PROVENANCE, indent=2))
    COMMON = ['--model', MODEL, '--revision', REVISION, '--precision', 'fp32', '--batch', '1', '--accumulation', '1', '--learning-rate', '1e-5', '--workers', '0', '--seed', '42']


In [ ]:
with stage('baseline'):
    command([sys.executable, '-u', str(TRAINER), 'evaluate', '--manifest', str(OUT / 'validation.json'), '--output', str(OUT / 'baseline')] + COMMON, 'baseline.log')
    baseline_run = json.loads((OUT / 'baseline/run.json').read_text())
    if baseline_run['resolved_revision'] != REVISION: raise ValueError('Baseline revision mismatch')


In [ ]:
with stage('training-eight-steps/post'):
    command([sys.executable, '-u', str(TRAINER), 'train', '--manifest', str(OUT / 'train.json'), '--validation', str(OUT / 'validation.json'), '--output', str(OUT / 'trained'), '--max-steps', '8', '--save-steps', '8', '--require-finite-training'] + COMMON, 'training.log')
    state = json.loads((OUT / 'trained/trainer_state.json').read_text())
    if state['global_step'] != 8: raise ValueError('Expected exactly eight optimizer steps')


In [ ]:
with stage('smoke-report'):
    report = dict(counts=COUNTS, provenance=PROVENANCE,
        baseline=json.loads((OUT / 'baseline/metrics.json').read_text()),
        post=json.loads((OUT / 'trained/metrics.json').read_text()),
        caveat='Smoke only, not quality evidence. Subset byte audit only; no global byte, near-duplicate, official-test byte or pretraining exclusion guarantee.',
        logs=['baseline.log', 'training.log'], output=str(OUT))
    (OUT / 'smoke-report.json').write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2), flush=True)
